In [ ]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

In [2]:
load_dotenv()  # Load environment variables from .env file
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [3]:
from typing import Literal, TypedDict, Optional, Annotated

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
from langchain_classic.prompts import MessagesPlaceholder


prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. Reply briefly when possible."
    ),
    MessagesPlaceholder(variable_name="messages"),
    ])

In [4]:
def chat_node(state: ChatState) -> ChatState:
    messages = state["messages"]
    chain = prompt_template | llm
    response = chain.invoke({"messages": messages})
    
    return {"messages": [response]}

In [5]:
graph = StateGraph(ChatState)

In [7]:
#add nodes
graph.add_node('chat_node', chat_node)

In [8]:
#add edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)